# 🏀 Exploration & build log — SBRO odds ingestion

**Goal:** crack the structure of the raw Sportsbook Reviews Online (SBRO) NCAA men's
basketball odds and turn it into one clean row per game.

**Source:** a single season page saved from
`sportsbookreviewsonline.com/scoresoddsarchives/ncaa-basketball-2021-22/`. The site
blocks automated requests, so the page is saved by hand into `data/raw/odds/` and parsed
offline (the raw file's provenance is logged in `data/raw/odds/MANIFEST.md`).

> The finished, reusable version of this logic lives in `src/ncaa_market/ingest/odds.py`, which is what
> the pipeline actually runs. 

In [25]:
from __future__ import annotations

import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Finds main project directory
def _repo_root() -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "config" / "settings.yaml").exists():
            return root
    raise FileNotFoundError(f"Could not locate repo root from cwd={Path.cwd()}")

_src = _repo_root() / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from ncaa_market.config import get_config, resolve_path

# Pandas read the HTML with generic columns (0,1,2..) and the real header is stored as the first data row which we will later drop.
RAW_COLUMNS = ["Date", "Rot", "VH", "Team", "1st", "2nd", "Final", "Open", "Close", "ML", "2H"]

## 1. First look at the raw page

Before I write any parsing logic, I want to see what I'm actually working with. I saved the SBRO 2021–22 season page as `data/raw/odds/ncaab_2021-22.html`, and we're going to load it with `pd.read_html()`.

In [26]:
from typing import Any


tables = pd.read_html(resolve_path("data/raw/odds/ncaab_2021-22.html"))
raw = tables[0]
print("tables on the page:", len(tables))
print("shape:", raw.shape)
print("columns:", list[Any](raw.columns))
raw.head(6)

tables on the page: 1
shape: (10963, 11)
columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,0,1,2,3,4,5,6,7,8,9,10
0,Date,Rot,VH,Team,1st,2nd,Final,Open,Close,ML,2H
1,1109,601,V,UCSanDiego,33,47,80,141.5,140,700,74
2,1109,602,H,California,37,30,67,15,14.5,-1100,7.5
3,1109,603,V,SoIllinois,37,29,66,7,6.5,-275,3
4,1109,604,H,ArkansasLR,34,35,69,139,137.5,235,73.5
5,1109,605,V,Akron,28,38,66,146.5,140.5,775,73


## 2. The format, decoded

Four things jump out of that raw table:

1. **The header is in row 0** — pandas read the real headers
   (`Date, Rot, VH, Team, 1st, 2nd, Final, Open, Close, ML, 2H`) as the first *data* row,
   which is why the columns came through as `0..10`.
2. **Two rows per game** — an away row (`VH = V`) then a home row (`VH = H`), paired by
   **consecutive `Rot` numbers** (601/602, 603/604…). Neutral-site games are marked `N`
   on *both* rows.
3. **`Open`/`Close` hold either the spread *or* the total**, separated by **magnitude** —
   the smaller number is the point spread, the larger is the game total.
4. **Spread sign comes from the moneyline** — lower ML = favorite. SBRO often puts the
   spread magnitude on the underdog's row, so row position alone can't tell you the sign.
   Pick'ems sometimes appear as `1.0` instead of `PK` when both MLs are even (e.g. −110/−110).

The functions below encode exactly these rules: parse the long table, then collapse each
pair into one game with the spread in **home-team convention** (negative ⇒ home favored).

In [27]:
# Takes weird sportsbook text and turns it into a float if possible.
def _to_line(x) -> float:
    s = str(x).strip().upper()
    if s in {"PK", "PICK", "EVEN", "EV"}:
        return 0.0
    try:
        return float(s)
    except ValueError:
        return float("nan")

# Properly handle dates which are stored as MMDD with no year (1109 = Nov 9, 101 = Jan 1), 
# while correctly handling the fact that an NCAA season spans two calendar years.
def _parse_date(mmdd, start_year: int, end_year: int) -> pd.Timestamp:
    s = str(mmdd).strip().split(".")[0].zfill(4)  # "101" -> "0101"
    try:
        month, day = int(s[:2]), int(s[2:])
        year = start_year if month >= 7 else end_year
        return pd.Timestamp(year=year, month=month, day=day)
    except (ValueError, TypeError):
        return pd.NaT

# Take the messy Sportsbook Review HTML file and turn it into a clean pandas DataFrame.
def load_season_long(html_path: str | Path, season_end: int) -> pd.DataFrame:
    start_year = season_end - 1
    path = resolve_path(html_path)
    if not path.exists():
        raise FileNotFoundError(f"SBRO HTML not found: {path}")

    raw = pd.read_html(path)[0] # Reads the first HTML table in the file.

    # Row 0 holds the real headers -> drop it and assign explicit, clean names.
    df = raw.iloc[1:].copy()
    df.columns = RAW_COLUMNS
    df = df.reset_index(drop=True)

    df["date"] = df["Date"].apply(lambda d: _parse_date(d, start_year, season_end))
    for col in ["Rot", "1st", "2nd", "Final", "ML"]: # Convert to numeric columns
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ["Open", "Close", "2H"]: # Clean sportsbook line columns
        df[col] = df[col].apply(_to_line)
    df["VH"] = df["VH"].astype(str).str.strip().str.upper() # Clean VH column
    df["Team"] = df["Team"].astype(str).str.strip() # Clean team names
    return df # Return the cleaned DataFrame

# Decode the spread and total from the Open/Close columns.
def _decode_spread_total(
    away_line: np.ndarray,
    home_line: np.ndarray,
    ml_away: np.ndarray,
    ml_home: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:

    spread = np.minimum(away_line, home_line)  # Spread is the smaller of the two lines
    total = np.maximum(away_line, home_line)  # Total is the larger of the two lines
    away_fav = ml_away < ml_home  # Away team is favored if its ML is less than the home team's ML
    home_fav = ml_home < ml_away  # Home team is favored if its ML is less than the away team's ML
    pickem = (np.abs(ml_away - ml_home) < 1) & (spread <= 1.0)  # Likely pick'em

    # Fallback: if ML is missing/equal (can't infer favorite), keep a usable sign
    # using SBRO row-position heuristic.
    spread_on_home_row = home_line < away_line
    row_signed = np.where(spread_on_home_row, -spread, spread)

    signed = np.where(
        pickem,
        0.0,
        np.where(away_fav, spread, np.where(home_fav, -spread, row_signed)),
    )
    return signed, total

# Collapse the two-rows-per-game into one row per game.
def collapse_to_games(long_df: pd.DataFrame, season_end: int) -> pd.DataFrame:

    # Split away and home rows
    away = long_df.iloc[0::2].reset_index(drop=True)  
    home = long_df.iloc[1::2].reset_index(drop=True)   

    # Create a new DataFrame with the collapsed data
    games = pd.DataFrame({
        "season": season_end,
        "date": away["date"],
        "away_team": away["Team"],
        "home_team": home["Team"],
        "away_score": away["Final"],
        "home_score": home["Final"],
        "ml_away": away["ML"],
        "ml_home": home["ML"],
        "neutral_site": ((away["VH"] == "N") | (home["VH"] == "N")).astype(int),
    })

    # Convert ML columns to numpy arrays for efficient processing
    ml_away = away["ML"].to_numpy(dtype="float64")
    ml_home = home["ML"].to_numpy(dtype="float64")
    for stage in ["Open", "Close"]: # Decode the spread and total
        spread, total = _decode_spread_total(
            away[stage].to_numpy(dtype="float64"),
            home[stage].to_numpy(dtype="float64"),
            ml_away,
            ml_home,
        )
        key = stage.lower()
        games[f"spread_{key}"] = spread
        games[f"total_{key}"] = total

    away_fav = ml_away < ml_home
    home_fav = ml_home < ml_away

    # Set the favorite team based on the ML
    games["favorite"] = np.where( 
        away_fav,
        games["away_team"],
        np.where(home_fav, games["home_team"], pd.NA),
    )
    return games

def validate_games(games: pd.DataFrame) -> None:

    # "Did we accidentally create a game where the away team and home team are the same?"
    if games["away_team"].eq(games["home_team"]).any():
        raise ValueError("Found games where away_team == home_team — pairing is wrong.")
    
    # Count the number of games which should match games.shape[0]
    n = len(games)

    # Count the number of games with bad dates
    bad_dates = int(games["date"].isna().sum())

    # Count the number of games missing the closing spread
    miss_close = int(games["spread_close"].isna().sum())

    # Count the number of games missing the score
    miss_score = int(games["home_score"].isna().sum() + games["away_score"].isna().sum())

    print(f"  validate: {n} games | bad dates={bad_dates} | "
          f"missing closing spread={miss_close} | rows missing score={miss_score}")

# Save the parsed season to the staging layer as Parquet.
def save_games(games: pd.DataFrame, season_end: int) -> Path:
    out_dir = resolve_path(get_config()["paths"]["staging"]) / "odds"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"odds_{season_end}.parquet"
    games.to_parquet(out_path, index=False)
    return out_path


def write_manifest(html_path: str | Path, season_end: int, source_url: str, n_games: int) -> None:
    """Append a provenance row for the raw download to data/raw/odds/MANIFEST.md."""
    path = resolve_path(html_path)
    raw_dir = resolve_path(get_config()["paths"]["raw"]) / "odds"
    raw_dir.mkdir(parents=True, exist_ok=True)
    manifest = raw_dir / "MANIFEST.md"
    sha = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    today = pd.Timestamp.today().date().isoformat()
    if not manifest.exists():
        manifest.write_text(
            "# Raw odds — provenance\n\n"
            "| Season | File | Source | Downloaded | Games | SHA256 |\n"
            "|---|---|---|---|---|---|\n",
            encoding="utf-8",
        )
    row = (f"| {season_end - 1}-{str(season_end)[2:]} | {path.name} | "
           f"{source_url} | {today} | {n_games} | {sha} |\n")
    with manifest.open("a", encoding="utf-8") as fh:
        fh.write(row)

# Our pipeline function.
# Full single-season odds ingest: load -> collapse -> validate -> save -> manifest.
def ingest_season(html_path: str | Path, season_end: int, source_url: str) -> pd.DataFrame:
    long_df = load_season_long(html_path, season_end)
    games = collapse_to_games(long_df, season_end)
    validate_games(games)
    out_path = save_games(games, season_end)
    write_manifest(html_path, season_end, source_url, len(games))
    print(f"  saved {len(games)} games -> {out_path}")
    return games

## 3. Verify, then run the full ingest

Confirm the rows really pair by consecutive
`Rot` numbers, spot-check the decoded spread/total/favorite on games we already know,
then run the full single-season ingest (validate → save → manifest).

In [28]:
# Load the raw HTML file and cleans dates, spreads, and strings -> numbers 
long_df = load_season_long("data/raw/odds/ncaab_2021-22.html", season_end=2022)

# Every game is two rows with consecutive rotation numbers.
first = long_df.iloc[0::2].reset_index(drop=True)
second = long_df.iloc[1::2].reset_index(drop=True)
print("rows even?", len(long_df) % 2 == 0)
print("consecutive-Rot pairs:", int((second["Rot"] - first["Rot"] == 1).sum()), "of", len(first))

# Spot-check the decode on three games we already know. [Matches]
games = collapse_to_games(long_df, season_end=2022)
display(games.head(3)[["away_team", "home_team", "away_score", "home_score",
                       "spread_close", "total_close", "favorite", "neutral_site"]])

# Run the full single-season ingest (validate -> save Parquet -> append manifest).
games = ingest_season(
    html_path="data/raw/odds/ncaab_2021-22.html",
    season_end=2022,
    source_url="https://www.sportsbookreviewsonline.com/scoresoddsarchives/ncaa-basketball-2021-22/",
)

rows even? True
consecutive-Rot pairs: 5481 of 5481


,away_team,home_team,away_score,home_score,spread_close,total_close,favorite,neutral_site
0,UCSanDiego,California,80.0,67.0,-14.5,140.0,California,0
1,SoIllinois,ArkansasLR,66.0,69.0,6.5,137.5,SoIllinois,0
2,Akron,OhioState,66.0,67.0,-16.5,140.5,OhioState,0


  validate: 5481 games | bad dates=0 | missing closing spread=5 | rows missing score=8
  saved 5481 games -> C:\Users\johnn\OneDrive\Desktop\batman\ncaa_market\data\staging\odds\odds_2022.parquet


In [29]:
miss = games[games["spread_close"].isna()]
print(len(miss), "games have no closing spread")

# THE diagnostic: do these games also lack an opening line and a moneyline?
print("...also missing OPEN spread:", int(miss["spread_open"].isna().sum()), "of", len(miss))
print("...also missing home moneyline:", int(miss["ml_home"].isna().sum()), "of", len(miss))

# A few rows as text (copy-friendly)
print(miss[["date", "away_team", "home_team", "spread_open", "ml_home"]].head(10).to_string())

5 games have no closing spread
...also missing OPEN spread: 3 of 5
...also missing home moneyline: 5 of 5
           date   away_team      home_team  spread_open  ml_home
1737 2021-12-18   NCCentral  DelawareState          NaN      NaN
3884 2022-02-10     UCDavis     UCSanDiego          NaN      NaN
4098 2022-02-14  SantaClara      PortlandU          8.0      NaN
4102 2022-02-14   HighPoint       Longwood         -5.5      NaN
4247 2022-02-17  CalPolySLO        UCDavis          NaN      NaN


## Conclusion

Single-season ingestion works end to end: raw HTML → typed long frame → one row per game
(spread/total/favorite decoded, home convention) → validated → saved to
`data/staging/odds/odds_2022.parquet`, with provenance in `data/raw/odds/MANIFEST.md`.

This logic was promoted to **`src/ncaa_market/ingest/odds.py`** (the version the pipeline
runs). **Next:** efficiency features — market error, cover outcome, and CLV — in
`src/ncaa_market/features/ats.py`, then scaling the ingest across all seasons.